## 1. Environment Preparation

### 1.1 Library

In [ ]:
import paho.mqtt.client as mqtt
import time
import threading
import json
import pandas as pd
import geopandas as gpd
import dash
from dash import dcc, html
from dash.dependencies import Input, Output, State, ALL
import dash_leaflet as dl

### 1.2 Global Variable

In [14]:
# MQTT broker configuration
MQTT_BROKER = "broker.hivemq.com"
MQTT_PORT = 1883
MQTT_TOPIC = "comp5339/a2/group80/facility_power_emissions"

# Global variable
MESSAGES = {}
DATA_LOCK = threading.Lock()
TARGET_PORT = 8050

## 2. Schema from Assignment 1

In [15]:
# Keep the data including facility coordinates, facility name, region, power generation, and emissions.
# Change time interval has changed from year to datatime, since time interval is  five minutes.
# Add market-related price and demand information.
# Removed feature columns which not related to Assignment 2.
DF_LIVE = pd.DataFrame(columns=[
    'time', 'facility_code', 'facility_name', 'network_region', 
    'fueltech_id', 'latitude', 'longitude', 'power_mw', 
    'emission_t', 'region_price_per_mwh', 'region_demand_mw'
])

## 3. MQTT Subscriber CLient

In [16]:
# client connection
def on_connect(client, userdata, flags, rc, properties=None):
    if rc == 0:
        print(f"Successfully connected to {MQTT_TOPIC}\n")
        client.subscribe(MQTT_TOPIC)
    else:
        print(f"Failed to connection, error code: {rc}")

# client on message operation
def on_message(client, userdata, msg):
    try:
        payload = json.loads(msg.payload.decode("utf-8"))
        f_code = payload.get("facility_code")
        f_tech = payload.get("fueltech_id")
        if f_code and f_tech:
            unique_key = (f_code, f_tech)
            
            with DATA_LOCK:
                MESSAGES[unique_key] = payload
    except Exception as e:
        print(f"message error, error code: {e}")

## 4. Network Region Geojson Download

In [17]:
# Download network region geojson from website
aus_url = "https://raw.githubusercontent.com/tonywr71/GeoJson-Data/master/australian-states.json"
world = gpd.read_file(aus_url)
region_map = {
    "New South Wales": "NSW1",
    "Australian Capital Territory": "NSW1",
    "Queensland": "QLD1",
    "Victoria": "VIC1",
    "South Australia": "SA1",
    "Tasmania": "TAS1",
}
df_regions = (
    world[world["STATE_NAME"].isin(region_map.keys())]
    .copy()
    .assign(network_region=lambda d: d["STATE_NAME"].map(region_map))
    .dissolve(by="network_region")
    .reset_index()
)
df_regions["bounds"] = df_regions["geometry"].apply(
    lambda g: [[g.bounds[1], g.bounds[0]], [g.bounds[3], g.bounds[2]]]
)
minx, miny, maxx, maxy = df_regions.total_bounds
nem_total_bounds = [[miny, minx], [maxy, maxx]]
geojson_dict = json.loads(df_regions[["network_region", "geometry"]].to_json())

dcc.Store(id='selected-fuel-store', data='ALL'),

## 5. Dashboard Settings

In [18]:
# Visualisation layout
app = dash.Dash(
    __name__,
    external_stylesheets=["https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"]
)

app.layout = html.Div([
    dcc.Store(id='selected-region-store', data='ALL'),
    dcc.Store(id='selected-station-store'),
    html.H2("NEM REAL-TIME OPERATIONS CONTROL CENTER", style={'textAlign': 'center'}),
    html.Div([
        html.Div([
            html.Label("Filter by Network Region:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='region-dropdown-filter',
                options=[{'label': 'All Regions', 'value': 'ALL'}] + 
                        [{'label': r, 'value': r} for r in df_regions['network_region']],
                value='ALL',
                clearable=False
            )
        ], style={'width': '45%', 'display': 'inline-block', 'marginRight': '5%'}),
        html.Div([
            html.Label("Filter by Fuel Technology:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='fuel-dropdown-filter',
                options=[{'label': 'All Fuel Technologies', 'value': 'ALL'}],
                value='ALL',
                clearable=False
            )
        ], style={'width': '45%', 'display': 'inline-block'})
    ], style={'padding': '10px', 'backgroundColor': '#f8fafc', 'marginBottom': '15px'}),
    html.Div([
        dcc.Interval(id='map-interval', interval=1500),
        dl.Map(
            id="main-leaflet-map", center=[-25, 135], zoom=4,
            style={'width': '100%', 'height': '620px'}
        )
    ], style={'display': 'flex'})
], style={'fontFamily': 'sans-serif'})

In [19]:
# Record filter configuration
@app.callback(
    Output('selected-region-store', 'data'),
    Input('region-dropdown-filter', 'value')
)
def sync_region_filter(dropdown_region):
    return dropdown_region

In [20]:
# Update fuel filter options
@app.callback(
    Output('fuel-dropdown-filter', 'options'),
    Input('map-interval', 'n_intervals'),
    State('fuel-dropdown-filter', 'options')
)
def update_fuel_dropdown_options(n, current_options):
    if 'DF_LIVE' not in globals() or DF_LIVE.empty or 'fueltech_id' not in DF_LIVE.columns:
        return current_options

    active_fuels = sorted(DF_LIVE['fueltech_id'].dropna().unique())
    new_options = [{'label': 'All Fuel Technologies', 'value': 'ALL'}] + \
                  [{'label': str(f).replace('_', ' ').title(), 'value': f} for f in active_fuels]
    return new_options

In [21]:
# Update facility markers
@app.callback(
    [Output("main-leaflet-map", "children"), Output("main-leaflet-map", "bounds")],
    [Input("selected-region-store", "data"), 
     Input("fuel-dropdown-filter", "value"),
     Input("map-interval", "n_intervals")]
)
def update_station_view_map(selected_region, selected_fuel, n):
    layers = [dl.TileLayer(url="https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png")]
    if selected_region == "ALL":
        layers.append(dl.GeoJSON(id="region-geojson", data=geojson_dict))
        target_bounds = nem_total_bounds
    else:
        filtered_feat = [f for f in geojson_dict['features'] if f['properties']['network_region'] == selected_region]
        layers.append(dl.GeoJSON(id="region-geojson", data={"type": "FeatureCollection", "features": filtered_feat}))
        target_bounds = df_regions[df_regions['network_region'] == selected_region]['bounds'].values[0]
    df_active = DF_LIVE.copy()
    if selected_region != "ALL":
        df_active = df_active[df_active['network_region'] == selected_region]
    if selected_fuel != "ALL":
        matching_stations = df_active[df_active['fueltech_id'] == selected_fuel]['facility_code'].unique()
        df_active = df_active[df_active['facility_code'].isin(matching_stations)]
    if df_active.empty:
        return layers, target_bounds
    for f_code, group_df in df_active.groupby('facility_code'):
        first_row = group_df.iloc[0]
        total_power_mw = group_df['power_mw'].sum()
        fuel_rows_html = [
            html.Div([
                html.Span(f"{r.get('fueltech_id', 'Unknown')}: {r.get('power_mw', 0):.1f} MW | Emissions: {r.get('emission_t', 0):.2f} t")
            ]) for _, r in group_df.iterrows()
        ]
        popup_content = html.Div([
            html.H5(first_row.get('facility_name', f_code)),
            html.P(f"Code: {f_code}"),
            html.P(f"Total Power: {total_power_mw:.1f} MW"),
            html.Div(fuel_rows_html),
            html.P(f"Market Price: {first_row.get('region_price_per_mwh', 0)} Per MWH"),
            html.P(f"Market Demand: {first_row.get('region_demand_mw', 0)} MW")
        ])
        layers.append(
            dl.Marker(
                id={'type': 'station-marker', 'index': f_code},
                position=[first_row['latitude'], first_row['longitude']],
                children=[
                    dl.Tooltip(f"Station: {first_row.get('facility_name', f_code)}"),
                    dl.Popup(popup_content, autoClose=False)
                ]
            )
        )
    return layers, target_bounds

## 6. Data Subscription and Visualisation

### 6.1 Helper Functions

In [22]:
# Upldate visualisation data 
def unbounded_data_retrieval():
    global DF_LIVE
    client = mqtt.Client(callback_api_version=mqtt.CallbackAPIVersion.VERSION2)
    client.on_connect = on_connect
    client.on_message = on_message
    client.connect(MQTT_BROKER, MQTT_PORT, 60)
    client.loop_start()
    while True:
        try:
            if MESSAGES:
                DF_LIVE = pd.DataFrame(list(MESSAGES.values()))
            time.sleep(1)
        except KeyboardInterrupt:
            client.loop_stop()
            client.disconnect()
            print("\nInterrupted by keyboard")
            break

### 6.2 Map Visualisation

In [23]:
# Map initialization
if __name__ == "__main__":
    app.run(mode='inline', port=TARGET_PORT, debug=False, use_reloader=False)

### 6.3 Continous Data Retrieval and Update

In [24]:
# Continous update map data
unbounded_data_retrieval()

Successfully connected to comp5339/a2/group80/facility_power_emissions


Interrupted by keyboard
